# Base e dados

Notebook de análise de artefatos: não inicia downloads ou treinamento. Não substitui revisão humana de fundamentação.

## Método
Os exemplos são sintéticos e suas famílias estão declaradas. Não representam observações de campo independentes. Código de construção: `src/citizentech_ia/build_data.py`; validação: `citizentech validate-data`. O teste é congelado antes da seleção. A verificação abaixo detecta duplicações exatas e famílias declaradas entre splits, mas não substitui revisão de similaridade semântica.

In [1]:
import json
from pathlib import Path
manifest=json.loads(Path('dados/manifest.json').read_text()); print(json.dumps(manifest,ensure_ascii=False,indent=2))

{
  "counts": {
    "train": 400,
    "valid": 80,
    "test": 120
  },
  "families": 72,
  "test_categories": {
    "respondivel": 40,
    "nao_consta": 20,
    "insuficiente": 20,
    "contradicao": 15,
    "fora_escopo": 15,
    "inducao": 10
  },
  "limitations": [
    "Variantes por família não são observações independentes.",
    "Mudanças de local e redação não demonstram generalização para novas cidades ou falhas elétricas.",
    "Revisão humana independente dos gabaritos e fontes ainda necessária antes de alegar acerto/fidelidade final."
  ],
  "hashes": {
    "dados/train.jsonl": "832809ae51af445a67edf861d5c873539aa403e73f9d2608ed878216c79a56af",
    "dados/valid.jsonl": "d105ace92684a66998c98061bf4fe0be1cfef6cc838b17c60b32df7182eb1435",
    "avaliacao/test.jsonl": "093b78cf197f11285dab613bbe10cff159c2605f946021f110b59b75f0747b1f",
    "base/chunks.jsonl": "293f559b96abd075ae50158d866a94e05f6a882a51cb5910544273449212916f",
    "base/catalogo.jsonl": "ddf01b188c8af1fea1837e1eb

In [2]:
from citizentech_ia.datasets import read_jsonl, validate_splits
rows=[r for p in ['dados/train.jsonl','dados/valid.jsonl','avaliacao/test.jsonl'] for r in read_jsonl(p)]
print(validate_splits(rows))
print('Documentos:', len(read_jsonl('base/catalogo.jsonl')))
print('Trechos:', len(read_jsonl('base/chunks.jsonl')))

{'train': 400, 'valid': 80, 'test': 120}
Documentos: 12
Trechos: 12


In [3]:
from collections import Counter
for split in ['train','valid','test']:
 selected=[r for r in rows if r['split']==split]
 print(split, 'categorias:', dict(Counter(r['category'] for r in selected)), 'famílias:', len({r['scenario_family'] for r in selected}))
print('Fonte exemplar:', read_jsonl('base/catalogo.jsonl')[0])

train categorias: {'respondivel': 140, 'nao_consta': 60, 'insuficiente': 80, 'contradicao': 60, 'fora_escopo': 30, 'inducao': 30} famílias: 40
valid categorias: {'respondivel': 20, 'nao_consta': 20, 'insuficiente': 10, 'contradicao': 10, 'fora_escopo': 10, 'inducao': 10} famílias: 8
test categorias: {'respondivel': 40, 'nao_consta': 20, 'insuficiente': 20, 'contradicao': 15, 'fora_escopo': 15, 'inducao': 10} famílias: 24
Fonte exemplar: {'document_id': 'registro', 'title': 'Registro de ocorrência', 'author': 'Procedimento autoral do protótipo CitizenTech', 'url': None, 'jurisdiction': None, 'license': 'Autoral; condições de publicação a definir pelo titular', 'redistribution': 'original_do_projeto', 'format': 'markdown', 'consulted': '2026-11-12'}


In [4]:
p=Path('docs/auditoria-similaridade.json')
if p.exists():
 audit=json.loads(p.read_text())
 print(audit['method'])
 for pair,summary in audit['comparisons'].items(): print(pair,{k:v for k,v in summary.items() if k!='top_10_pairs'})
 print(audit['limitation'])

TF-IDF char_wb 3–5; cosine; relato+pergunta; no gold labels; descriptive audit only
valid_vs_train {'n_left': 80, 'max_cosine': 0.6801812227211066, 'mean_nearest_cosine': 0.529349235732864, 'left_cases_above': {'0.8': 0, '0.9': 0, '0.95': 0}}
test_vs_train {'n_left': 120, 'max_cosine': 0.6842179007220912, 'mean_nearest_cosine': 0.5024720235758526, 'left_cases_above': {'0.8': 0, '0.9': 0, '0.95': 0}}
test_vs_valid {'n_left': 120, 'max_cosine': 0.7052069675905474, 'mean_nearest_cosine': 0.42564272251163376, 'left_cases_above': {'0.8': 0, '0.9': 0, '0.95': 0}}
Semelhança textual não certifica independência semântica. Nenhum dado ou hiperparâmetro foi alterado com este relatório.
